# Refactor histogram, binning of MNase-seq data
February 13, 2024

Upon realization that the MOSEK solver in cvxpy allows for fast optimization for higher resolutions. It has become clear that deconvolving the chromatin at higher resolutions becomes possible.

However, in order to take advantage of these higher resolutions, the data will need to be blurred prior to binning. Currently there is not a way to do that easily. So, the task here is to create an exact histogram/img of the chromatin data, blur appropriate, then bin/downscale to the desired resolution for deconvolution.

### Tasks
1. Create an exact histogram of the MNase-seq data
2. Blur the histogram appropriately
3. Identify the +1 nucleosome location and center our histogram around this location
    - Subset the histogram and define the genomic span for this updated region.
4. Normalize the data
5. Bin/downscale to the appropriate resolution

### Notes
- Where do we normalize the data?
    - Before or after blurring?


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from cc_src.chromatin_model import ChromatinModel
from src.config import load_yl_replicate1_rg1_alpha_vst_config

gene_name = 'CLB2'
config1 = load_yl_replicate1_rg1_alpha_vst_config()
chromatin_model = ChromatinModel(config1)
chromatin_model.load_mnase_gene(gene_name, 1)

Loading MNase reads for CLB2...Done.
The histogram shape around the TSS is: (3, 9)
The shape of the flattened grid to be deconvolved is: (16, 27)
Applying normalization using scaling matrix: output/mnase/rep1_len_scaling_3len_bins.csv
The shape of the flattened grid to be deconvolved is: (16, 27)


In [3]:
# A large MNase-seq window -1000, +1000
chromatin_model.mnase_span





(770273, 772273)

In [5]:
# Let's actually pre-compute the +1 nucleosome locations
# to free up the chromatin model from not having to do this.

chromatin_model.geneset

,gene,chr,cat,start,stop,strand,classification,length,TSS,PAS,manually_curated,promoter_start,promoter_end,gene_body_start,gene_body_end
orf_name,,,,,,,,,,,,,,,
YAL068C,PAU8,1,gene,1807,2169,-,Verified,362,2169,NaN,NaN,2169.0,2469.0,1669.0,2169.0
YAL067W-A,YAL067W-A,1,gene,2480,2707,+,Uncharacterized,227,2480,NaN,NaN,2180.0,2480.0,2480.0,2980.0
YAL067C,SEO1,1,gene,7235,9016,-,Verified,1781,9016,NaN,NaN,9016.0,9316.0,8516.0,9016.0
YAL065C,YAL065C,1,gene,11565,11951,-,Uncharacterized,386,11951,NaN,NaN,11951.0,12251.0,11451.0,11951.0
YAL064W-B,YAL064W-B,1,gene,12046,12426,+,Uncharacterized,380,12046,NaN,NaN,11746.0,12046.0,12046.0,12546.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YPR200C,ARR2,16,gene,939279,939671,-,Verified,392,939779,939195.0,False,939779.0,940079.0,939279.0,939779.0
YPR201W,ARR3,16,gene,939922,941136,+,Verified,1214,939858,NaN,False,939558.0,939858.0,939858.0,940358.0
YPR202W,YPR202W,16,gene,943032,943896,+,Uncharacterized,864,942768,NaN,False,942468.0,942768.0,942768.0,943268.0


In [6]:
chromatin_model.find_max_plusOne_pos()

771273